# 07 — Visualización del árbol k-d y la poda branch and bound

Construye un árbol k-d pequeño sobre una submuestra de ULB, clasifica una transacción de consulta y **registra qué ramas visita y cuáles poda**. Genera:
1. El diagrama del árbol (nodos visitados en azul, podados en rojo).
2. La partición del espacio 2D con la consulta y las regiones podadas sombreadas.

Requiere el 01 (`datos_procesados/ulb.npz`). Corre todas las celdas.

In [ ]:
import os, datetime as _dt

def iniciar_ejecucion(nombre, config=None, base="resultados"):
    os.makedirs(base, exist_ok=True)
    nums = []
    for d in os.listdir(base):
        if d.startswith("ejecucion_") and os.path.isdir(os.path.join(base, d)):
            try: nums.append(int(d.split("_")[1]))
            except: pass
    n = max(nums)+1 if nums else 1
    carpeta = os.path.join(base, f"ejecucion_{n}")
    os.makedirs(carpeta)
    with open(os.path.join(carpeta, "config.txt"), "w", encoding="utf-8") as f:
        f.write("notebook : " + str(nombre) + "\n")
        f.write("fecha    : " + _dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S") + "\n")
        if config:
            for k, v in config.items():
                f.write(str(k) + " : " + str(v) + "\n")
    print("[Ejecucion " + str(n) + "] Resultados en: " + carpeta)
    return carpeta

def log(texto, carpeta):
    print(texto)
    with open(os.path.join(carpeta, "resultados.txt"), "a", encoding="utf-8") as f:
        f.write(str(texto) + "\n")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

SEMILLA   = 42
N_PUNTOS  = 256
PROFUNDIDAD = 5
DIM_X, DIM_Y = 0, 1
K = 1

In [ ]:
class Nodo:
    __slots__=("eje","corte","izq","der","puntos","id","caja")
    def __init__(s): s.eje=s.corte=s.izq=s.der=s.puntos=s.id=s.caja=None

def construir(X, ids, prof, caja, contador=[0]):
    n=Nodo(); n.id=contador[0]; contador[0]+=1; n.caja=caja
    if prof==0 or len(ids)<=4:
        n.puntos=ids; return n
    eje=prof%2
    vals=X[ids,eje]; med=np.median(vals); n.eje=eje; n.corte=med
    li=ids[vals<=med]; ld=ids[vals>med]
    if len(li)==0 or len(ld)==0: n.puntos=ids; return n
    cl=list(caja); cl[eje*2+1]=med
    cd=list(caja); cd[eje*2]=med
    n.izq=construir(X,li,prof-1,cl,contador)
    n.der=construir(X,ld,prof-1,cd,contador)
    return n

def dist_min_caja(q, caja):
    dx=max(caja[0]-q[0],0,q[0]-caja[1]); dy=max(caja[2]-q[1],0,q[1]-caja[3])
    return np.hypot(dx,dy)

def buscar(nodo, q, X, mejor, visitados, podados):
    """Branch and bound: si la distancia minima a la caja del nodo excede
    la mejor distancia actual, se PODA la rama completa."""
    if nodo is None: return mejor
    if dist_min_caja(q, nodo.caja) > mejor[1]:
        podados.append(nodo); return mejor
    visitados.append(nodo)
    if nodo.puntos is not None:
        for i in nodo.puntos:
            d=np.hypot(*(X[i]-q))
            if d<mejor[1]: mejor=(i,d)
        return mejor
    primero,segundo = (nodo.izq,nodo.der) if q[nodo.eje]<=nodo.corte else (nodo.der,nodo.izq)
    mejor=buscar(primero,q,X,mejor,visitados,podados)
    mejor=buscar(segundo,q,X,mejor,visitados,podados)
    return mejor

def contar(nodo):
    if nodo is None: return 0
    return 1+contar(nodo.izq)+contar(nodo.der)

In [ ]:
carpeta=iniciar_ejecucion("07_arbol_bb",{"n_puntos":N_PUNTOS,"profundidad":PROFUNDIDAD})
d=np.load("datos_procesados/ulb.npz",allow_pickle=True)
Xtr,ytr,Xte,yte=d["X_train"],d["y_train"],d["X_test"],d["y_test"]

rng=np.random.default_rng(SEMILLA)
idx=rng.choice(len(Xtr),N_PUNTOS,replace=False)
X2=Xtr[idx][:,[DIM_X,DIM_Y]].astype(float)
y2=ytr[idx]

caja=[X2[:,0].min()-0.5,X2[:,0].max()+0.5,X2[:,1].min()-0.5,X2[:,1].max()+0.5]
raiz=construir(X2,np.arange(N_PUNTOS),PROFUNDIDAD,caja)

qi=np.where(yte==1)[0][0]
q=Xte[qi][[DIM_X,DIM_Y]].astype(float)

visitados,podados=[],[]
mejor=buscar(raiz,q,X2,(None,np.inf),visitados,podados)
total=contar(raiz)
podados_sub=sum(contar(n) for n in podados)
log(f"Nodos del arbol: {total}",carpeta)
log(f"Nodos visitados: {len(visitados)}  |  Subarboles podados: {len(podados)} "
    f"({podados_sub} nodos eliminados sin explorar, {100*podados_sub/total:.1f}%)",carpeta)
log(f"Vecino mas cercano: punto {mejor[0]}, distancia {mejor[1]:.3f}, "
    f"clase {'FRAUDE' if y2[mejor[0]]==1 else 'legitima'} -> clasificacion k=1",carpeta)

In [ ]:
def posiciones(nodo,x0,x1,nivel,pos):
    if nodo is None: return
    pos[nodo.id]=((x0+x1)/2,-nivel)
    if nodo.puntos is None:
        m=(x0+x1)/2
        posiciones(nodo.izq,x0,m,nivel+1,pos)
        posiciones(nodo.der,m,x1,nivel+1,pos)

pos={}; posiciones(raiz,0,1,0,pos)
vis={n.id for n in visitados}; pod={n.id for n in podados}

def marcar_sub(n,s):
    if n is None: return
    s.add(n.id); marcar_sub(n.izq,s); marcar_sub(n.der,s)
pod_todo=set()
for n in podados: marcar_sub(n,pod_todo)

fig,ax=plt.subplots(figsize=(11,5.8))
def dibujar(n):
    if n is None: return
    for h in (n.izq,n.der):
        if h is not None:
            x1,y1=pos[n.id]; x2,y2=pos[h.id]
            activo = (h.id in vis) or (h.id in pod)
            ax.plot([x1,x2],[y1,y2],color="#444444" if activo else "#cccccc",
                    lw=2.0 if activo else 0.8, zorder=1)
            dibujar(h)
dibujar(raiz)
for nid,(x,y) in pos.items():
    if nid in pod:        col,ec,s2="#e74c3c","#7b241c",170
    elif nid in pod_todo: col,ec,s2="#f8cbc4","#f8cbc4",90
    elif nid in vis:      col,ec,s2="#2e86c1","#1b4f72",170
    else:                 col,ec,s2="#d5d8dc","#aab7b8",90
    ax.scatter([x],[y],s=s2,c=col,edgecolors=ec,zorder=2)
ax.set_axis_off()
ax.set_title(f"Recorrido branch and bound del árbol k-d "
             f"({len(visitados)} nodos visitados, {100*podados_sub/total:.0f}% del árbol podado)")
leg=[mpatches.Patch(color="#2e86c1",label="Visitado"),
     mpatches.Patch(color="#e74c3c",label="Podado (raíz del subárbol)"),
     mpatches.Patch(color="#f8cbc4",label="Nunca explorado (bajo un nodo podado)")]
ax.legend(handles=leg,loc="upper center",bbox_to_anchor=(0.5,-0.03),ncol=3,frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(carpeta,"arbol_poda.png"),dpi=150,bbox_inches="tight"); plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(8,6.8))
def cajas(n):
    if n is None: return
    if n.id in pod:
        ax.add_patch(plt.Rectangle((n.caja[0],n.caja[2]),n.caja[1]-n.caja[0],
                     n.caja[3]-n.caja[2],facecolor="#e74c3c",alpha=0.15,edgecolor="#e74c3c",
                     lw=0.8,zorder=0))
        return
    if n.puntos is None:
        if n.eje==0: ax.plot([n.corte]*2,[n.caja[2],n.caja[3]],color="#999999",lw=0.9,zorder=1)
        else:        ax.plot([n.caja[0],n.caja[1]],[n.corte]*2,color="#999999",lw=0.9,zorder=1)
        cajas(n.izq); cajas(n.der)
cajas(raiz)
ax.scatter(X2[y2==0,0],X2[y2==0,1],s=14,c="#5dade2",label="Legítima",zorder=2)
ax.scatter(X2[y2==1,0],X2[y2==1,1],s=30,c="#f39c12",edgecolors="#9c640c",label="Fraude",zorder=3)
ax.scatter([q[0]],[q[1]],s=200,marker="*",c="#c0392b",edgecolors="k",label="Consulta",zorder=5)
vx,vy=X2[mejor[0]]
ax.scatter([vx],[vy],s=90,facecolors="none",edgecolors="#1e8449",lw=2.2,
           label="Vecino más cercano",zorder=4)
ax.set_xlabel(f"Característica {DIM_X+1}"); ax.set_ylabel(f"Característica {DIM_Y+1}")
ax.set_title("Partición k-d: las regiones sombreadas se descartan sin calcular distancias")
ax.legend(loc="upper left",fontsize=9,framealpha=0.9)
ax.set_xlim(caja[0],caja[1]); ax.set_ylim(caja[2],caja[3])

axz=ax.inset_axes([0.66,0.06,0.32,0.32])
r=0.15
axz.scatter(X2[y2==0,0],X2[y2==0,1],s=26,c="#5dade2",zorder=2)
axz.scatter(X2[y2==1,0],X2[y2==1,1],s=40,c="#f39c12",edgecolors="#9c640c",zorder=3)
axz.scatter([q[0]],[q[1]],s=260,marker="*",c="#c0392b",edgecolors="k",zorder=5)
axz.scatter([vx],[vy],s=120,facecolors="none",edgecolors="#1e8449",lw=2.2,zorder=4)
axz.add_patch(plt.Circle(q,mejor[1],fill=False,color="#c0392b",ls="--",lw=1.4,zorder=4))
axz.set_xlim(q[0]-r,q[0]+r); axz.set_ylim(q[1]-r,q[1]+r)
axz.set_xticks([]); axz.set_yticks([])
axz.set_title("Zoom a la consulta",fontsize=8)
ax.indicate_inset_zoom(axz,edgecolor="#555555")
plt.tight_layout()
plt.savefig(os.path.join(carpeta,"particion_poda.png"),dpi=150,bbox_inches="tight"); plt.show()
log(f"\nFiguras guardadas en: {carpeta}",carpeta)